# D-TEST: Deterministic Deep-Linear Depth-Scaling Proxy

This notebook is the presentation companion to `run.py` in the same directory. It uses the **same computation** as the script and is intentionally narrower and more honest than the previous narrative framing.

## Pair identity and truthful scope

- **Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/D-TEST_depth_exponent_scaling/run.py`
- **Experiment type:** deterministic toy deep-linear comparison of momentum SGD versus Muon-style momentum with Newton–Schulz gradient orthogonalization.
- **Primary metric:** semilog fit of the final-step loss-advantage ratio `loss_SGD / loss_Muon` versus depth.
- **Secondary diagnostics:** heuristic stable SGD learning rates, loss trajectories, condition-number trajectories, and steps-to-threshold speed metrics.

## What this notebook does *not* claim

This notebook does **not** present a proof of complexity-class separation. It does not provide asymptotic guarantees, uncertainty estimates across seeds, or a true Hessian spectral calculation. Its role is to make the deterministic script-level experiment reproducible, interpretable, and academically self-explanatory.


In [ ]:
import importlib.util
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')


def locate_experiment_dir(start: Path) -> Path:
    relative = Path('experiments') / 'Tier_1_Core_Mechanism_Tests' / 'D-TEST_depth_exponent_scaling'
    candidates = []
    for root in [start, *start.parents]:
        candidates.append(root)
        candidates.append(root / relative)

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'run.py').exists() and (candidate / 'run_experiment.ipynb').exists():
            return candidate

    raise FileNotFoundError('Could not locate the D-TEST experiment directory containing run.py and run_experiment.ipynb.')


EXPERIMENT_DIR = locate_experiment_dir(Path.cwd())
RUN_PY = EXPERIMENT_DIR / 'run.py'
NOTEBOOK_PATH = EXPERIMENT_DIR / 'run_experiment.ipynb'
REPO_ROOT = EXPERIMENT_DIR.parents[2]
RELATIVE_SCRIPT = RUN_PY.relative_to(REPO_ROOT)
RELATIVE_NOTEBOOK = NOTEBOOK_PATH.relative_to(REPO_ROOT)

spec = importlib.util.spec_from_file_location('dtest_depth_scaling_run', RUN_PY)
dtest_run = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dtest_run)

print(f'Experiment directory: {EXPERIMENT_DIR}')
print(f'Loaded script module from: {RUN_PY}')
print(f'Python: {platform.python_version()}')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'Matplotlib: {plt.matplotlib.__version__}')


## Reproducibility, execution path, and runtime

The notebook now imports the script helpers directly instead of duplicating the experiment logic. This is the key alignment choice for the pair: **`run.py` is the single source of truth for the numerical experiment**, while the notebook focuses on presentation, figures, interpretation, and limitations.

The default script entrypoint remains:

```bash
python experiments/Tier_1_Core_Mechanism_Tests/D-TEST_depth_exponent_scaling/run.py
```

Optional structured output is available from the script via:

```bash
python experiments/Tier_1_Core_Mechanism_Tests/D-TEST_depth_exponent_scaling/run.py --save-json path/to/results.json
```

The code cell below runs the default deterministic experiment through the imported helper API and records runtime in the current environment.


In [ ]:
t0 = time.perf_counter()
results = dtest_run.run_full_experiment(verbose=False)
runtime_seconds = time.perf_counter() - t0

config = results['config']
primary = results['analyses']['primary_depth_fit']
time_scaling = results['analyses']['time_scaling']
condition_analysis = results['analyses']['condition_numbers']
verdict = results['analyses']['verdict']
seeds = results['seeds']
problem_data = results['problem_data']


def none_to_nan(value):
    return np.nan if value is None else value


summary_records = []
threshold_records = []
for depth in config['depths']:
    depth_result = results['per_depth'][depth]
    threshold_map = {entry['fraction']: entry for entry in depth_result['threshold_steps']}
    summary_records.append({
        'depth': depth,
        'heuristic_sgd_lr': depth_result['lr_sgd'],
        'curvature_proxy': depth_result['lr_search']['lambda_proxy'],
        'theory_lr_before_cap': depth_result['lr_search']['lr_theory'],
        'final_loss_sgd': depth_result['final_loss_sgd'],
        'final_loss_muon': depth_result['final_loss_muon'],
        f'advantage_step_{primary["final_measurement_step"]}': depth_result['advantage_ratios'][primary['final_measurement_step']],
        'sgd_instability_step': none_to_nan(depth_result['instability']['sgd_step']),
        'muon_instability_step': none_to_nan(depth_result['instability']['muon_step']),
        'sgd_step_to_50pct': none_to_nan(threshold_map[0.5]['sgd_step']),
        'muon_step_to_50pct': none_to_nan(threshold_map[0.5]['muon_step']),
        'sgd_step_to_10pct': none_to_nan(threshold_map[0.1]['sgd_step']),
        'muon_step_to_10pct': none_to_nan(threshold_map[0.1]['muon_step']),
        'sgd_step_to_1pct': none_to_nan(threshold_map[0.01]['sgd_step']),
        'muon_step_to_1pct': none_to_nan(threshold_map[0.01]['muon_step']),
    })
    for entry in depth_result['threshold_steps']:
        threshold_records.append({
            'depth': depth,
            'fraction': entry['fraction'],
            'threshold_label': f"{int(round(entry['fraction'] * 100))}% of initial loss",
            'optimizer': 'SGD',
            'step': none_to_nan(entry['sgd_step']),
        })
        threshold_records.append({
            'depth': depth,
            'fraction': entry['fraction'],
            'threshold_label': f"{int(round(entry['fraction'] * 100))}% of initial loss",
            'optimizer': 'Muon',
            'step': none_to_nan(entry['muon_step']),
        })

summary_df = pd.DataFrame(summary_records)
threshold_df = pd.DataFrame(threshold_records)

print(f'Experiment runtime in this notebook kernel: {runtime_seconds:.2f} s')
print(f'Primary verdict label: {verdict["label"]}')


## Reproducibility summary

This section records counterpart identity, default seeds, runtime, and the exact deterministic configuration being exercised. The tables are intended to make the notebook legible as a standalone artifact without overstating the scope of the experiment.


In [ ]:
repro_df = pd.DataFrame([
    ('Notebook path', str(RELATIVE_NOTEBOOK)),
    ('Counterpart script path', str(RELATIVE_SCRIPT)),
    ('Exact default CLI command', f'python {RELATIVE_SCRIPT}'),
    ('Notebook execution cwd', str(Path.cwd())),
    ('Runtime in this kernel (s)', round(runtime_seconds, 3)),
    ('Problem seed', seeds['problem_seed']),
    ('LR-search seed', seeds['lr_search_seed']),
    ('Initialization rule', seeds['init_seed_rule']),
    ('Target shape', tuple(problem_data['target_shape'])),
    ('Input batch shape', tuple(problem_data['input_shape'])),
    ('Scope', results['scope']),
], columns=['Field', 'Value'])

config_df = pd.DataFrame([
    ('input_dim', config['input_dim']),
    ('hidden_dim', config['hidden_dim']),
    ('output_dim', config['output_dim']),
    ('batch_size', config['batch_size']),
    ('num_steps', config['num_steps']),
    ('depths', config['depths']),
    ('measurement_steps', config['measurement_steps']),
    ('lr_muon', config['lr_muon']),
    ('momentum', config['momentum']),
    ('ns_iters', config['ns_iters']),
    ('condition_measurement_interval', config['condition_measurement_interval']),
    ('analysis_depth', config['analysis_depth']),
    ('threshold_fractions', config['threshold_fractions']),
], columns=['Config key', 'Value'])

display(repro_df)
display(config_df)


## Per-depth summary table

Before plotting, it is useful to inspect the structured raw results exposed by the script. The table below includes the heuristic stable SGD learning rate, the final losses, the final-step advantage ratio, and the new steps-to-threshold speed metrics derived directly from the already-computed loss trajectories.


In [ ]:
display(
    summary_df.round({
        'heuristic_sgd_lr': 6,
        'curvature_proxy': 6,
        'theory_lr_before_cap': 6,
        'final_loss_sgd': 6,
        'final_loss_muon': 6,
        f'advantage_step_{primary["final_measurement_step"]}': 4,
    })
)


## Loss trajectories by depth

The figure below presents the raw loss curves for both optimizers across the full depth sweep. A log-scaled y-axis is used because the absolute loss levels vary substantially by depth. This figure is the most direct check that the fixed-horizon advantage ratios are grounded in the full trajectories rather than in a single endpoint alone.


In [ ]:
num_depths = len(config['depths'])
ncols = 2
nrows = int(np.ceil(num_depths / ncols))
steps = np.arange(config['num_steps'] + 1)

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.6 * nrows), sharex=True)
axes = np.array(axes).reshape(-1)

for ax, depth in zip(axes, config['depths']):
    depth_result = results['per_depth'][depth]
    loss_sgd = np.maximum(np.array(depth_result['losses_sgd'], dtype=float), 1e-16)
    loss_muon = np.maximum(np.array(depth_result['losses_muon'], dtype=float), 1e-16)
    ax.plot(steps, loss_sgd, color='tab:orange', linewidth=2, label='SGD')
    ax.plot(steps, loss_muon, color='tab:blue', linewidth=2, label='Muon')
    ax.set_yscale('log')
    ax.set_title(f'Depth L={depth}')
    ax.set_xlabel('Optimization step')
    ax.set_ylabel('Loss')
    final_adv = depth_result['advantage_ratios'][primary['final_measurement_step']]
    ax.text(
        0.03,
        0.03,
        f'Adv@{primary["final_measurement_step"]} = {final_adv:.2f}x\nSGD LR = {depth_result["lr_sgd"]:.4g}',
        transform=ax.transAxes,
        fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
    )

for ax in axes[num_depths:]:
    ax.axis('off')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=True)
fig.suptitle('Loss trajectories across depths: deterministic SGD vs Muon comparison', y=1.02, fontsize=14)
fig.tight_layout()
plt.show()


**Interpretation.** The trajectories show that the comparison is not a trivial one-step artifact: both optimizers make progress, but the fixed-horizon separation generally becomes more favorable to Muon as depth increases. At the same time, the low-depth cases show that the story is not monotone in every early-time detail; for example, SGD can reach loose loss thresholds faster at shallow depth even when Muon finishes with a smaller loss by step 300.


## Heuristic stable SGD learning rates

The script does **not** globally optimize SGD learning rates. Instead, it uses a depth-dependent heuristic: start from a curvature-proxy-based upper bound, cap it at `0.1`, and repeatedly halve it until a short validation run is stable. The figure and table below present these selected learning rates honestly as a heuristic stability baseline.


In [ ]:
lr_view = summary_df[['depth', 'heuristic_sgd_lr', 'theory_lr_before_cap', 'curvature_proxy']].copy()

fig, ax1 = plt.subplots(figsize=(8, 4.8))
ax1.semilogy(lr_view['depth'], lr_view['heuristic_sgd_lr'], 'o-', linewidth=2, label='Selected heuristic stable SGD LR', color='tab:orange')
ax1.semilogy(lr_view['depth'], np.minimum(lr_view['theory_lr_before_cap'], config['sgd_lr_cap']), '--', linewidth=2, label='Initial capped theory guess', color='tab:gray')
ax1.set_xlabel('Depth L')
ax1.set_ylabel('Learning rate (log scale)')
ax1.set_title('Heuristic stable SGD learning rate versus depth')
ax1.legend(loc='best')
plt.show()

display(lr_view.round(6))


**Interpretation.** The heuristic stable SGD learning rate shrinks sharply with depth, especially once the curvature proxy grows beyond the cap-controlled shallow-depth regime. This is an empirical feature of the chosen baseline and should be read as such: it helps contextualize the optimizer comparison, but it is not evidence that SGD has been globally tuned in any optimal sense.


## Primary analysis: semilog depth-fit of final-step advantage

The notebook's primary quantitative summary matches the script: take the Muon/SGD loss advantage at the final measurement step, compute `log(advantage)`, and fit a line versus depth. A clean fit is **suggestive** of depth-dependent multiplicative structure in this toy setting, but by itself it is still not a proof of asymptotic or complexity-theoretic behavior.


In [ ]:
depths_valid = np.array(primary['valid_depths'], dtype=float)
advantages_valid = np.array(primary['valid_advantages'], dtype=float)
predicted_advantages = np.array(primary['predicted_advantages'], dtype=float)
residuals = np.array(primary['residuals'], dtype=float)

fig, (ax_top, ax_bottom) = plt.subplots(
    2,
    1,
    figsize=(8, 8),
    sharex=True,
    gridspec_kw={'height_ratios': [3, 1]}
)

ax_top.semilogy(depths_valid, advantages_valid, 'o', markersize=8, color='tab:blue', label='Measured final-step advantage')
ax_top.semilogy(depths_valid, predicted_advantages, '-', linewidth=2, color='tab:red', label='Least-squares semilog fit')
ax_top.set_ylabel(f'Advantage at step {primary["final_measurement_step"]} (log scale)')
ax_top.set_title('Primary depth-scaling proxy: final-step advantage versus depth')
ax_top.legend(loc='best')
ax_top.text(
    0.03,
    0.05,
    f"log(adv) = {primary['slope']:.4f} · L + ({primary['intercept']:.4f})\nR² = {primary['r_squared']:.4f}\nBase per added layer = {primary['exp_base_per_layer']:.4f}",
    transform=ax_top.transAxes,
    fontsize=10,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85),
)

ax_bottom.axhline(0.0, color='black', linewidth=1)
ax_bottom.plot(depths_valid, residuals, 'o-', color='tab:purple', linewidth=2)
ax_bottom.set_xlabel('Depth L')
ax_bottom.set_ylabel('Residual\n(log-space)')
ax_bottom.set_title('Fit residuals')

fig.tight_layout()
plt.show()

fit_summary_df = pd.DataFrame({
    'depth': primary['valid_depths'],
    'measured_advantage': primary['valid_advantages'],
    'predicted_advantage': primary['predicted_advantages'],
    'log_residual': primary['residuals'],
    'sgd_trainable': [primary['trainable_by_depth'][depth] for depth in primary['valid_depths']],
})

display(fit_summary_df.round(4))


**Interpretation.** In the current deterministic run, the final-step advantage grows strongly enough with depth to yield a high-`R²` semilog fit. However, the fit is not perfectly monotone across shallow depths, and the evidence remains limited to seven deterministic points in a fixed 300-step horizon. The honest conclusion is therefore: **strong toy-scale empirical support for the semilog proxy metric, not a proof of a complexity separation.**


## Threshold-step speed metrics

To complement the endpoint-based advantage ratio, the script now reports the first step at which each optimizer reaches selected fractions of the **initial** SGD loss. This is a lightweight but serious speed metric derived from the already-computed trajectories, and it helps distinguish loose-threshold behavior from more stringent convergence behavior.


In [ ]:
threshold_levels = sorted(threshold_df['fraction'].unique(), reverse=True)
fig, axes = plt.subplots(1, len(threshold_levels), figsize=(15, 4.4), sharey=True)
if len(threshold_levels) == 1:
    axes = [axes]

for ax, fraction in zip(axes, threshold_levels):
    subset = threshold_df[threshold_df['fraction'] == fraction]
    for optimizer, color in [('SGD', 'tab:orange'), ('Muon', 'tab:blue')]:
        opt_subset = subset[subset['optimizer'] == optimizer]
        ax.plot(opt_subset['depth'], opt_subset['step'], 'o-', linewidth=2, color=color, label=optimizer)
    ax.set_title(f'First step to {int(round(fraction * 100))}% of initial loss')
    ax.set_xlabel('Depth L')
    ax.set_ylabel('Step')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=True)
fig.suptitle('Threshold-step speed metrics derived from the loss trajectories', y=1.05, fontsize=14)
fig.tight_layout()
plt.show()

display(
    threshold_df.pivot_table(index=['depth', 'threshold_label'], columns='optimizer', values='step').reset_index()
)

muon_better_records = []
for fraction in sorted(threshold_df['fraction'].unique()):
    subset = threshold_df[threshold_df['fraction'] == fraction]
    pivot = subset.pivot(index='depth', columns='optimizer', values='step')
    comparable = pivot.dropna()
    muon_better = int((comparable['Muon'] < comparable['SGD']).sum())
    sgd_better = int((comparable['SGD'] < comparable['Muon']).sum())
    muon_better_records.append({
        'threshold_fraction': fraction,
        'depths_compared': int(len(comparable)),
        'depths_where_muon_is_faster': muon_better,
        'depths_where_sgd_is_faster': sgd_better,
    })

display(pd.DataFrame(muon_better_records))


**Interpretation.** These threshold metrics show a more nuanced picture than the final-step advantage ratio alone. SGD often reaches **looser** thresholds sooner at shallow depth, while Muon becomes more competitive or faster on **stricter** thresholds as depth increases. This is precisely why the notebook now presents both endpoint and trajectory-derived metrics rather than relying on a single headline number.


## Condition-number diagnostic at the representative depth

The previous narrative overclaimed the condition-number mechanism by suggesting that the product Lyapunov exponent should exceed the sum of individual layer exponents. That framing is incorrect. For matrix products,

\[
\kappa\!\left(\prod_{\ell=1}^{L} W_\ellight) \leq \prod_{\ell=1}^{L} \kappa(W_\ell),
\]

so after taking logs,

\[
\log \kappa(	ext{product}) \leq \sum_{\ell=1}^{L} \log \kappa(W_\ell).
\]

The correct diagnostic is therefore not whether the product grows *faster* than the sum, but whether the observed trajectories respect this bound and how the growth patterns compare across optimizers.


In [ ]:
analysis_depth = condition_analysis['analysis_depth']
measurement_steps = np.array(condition_analysis['measurement_steps'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=True)
for optimizer, color in [('sgd', 'tab:orange'), ('muon', 'tab:blue')]:
    entry = condition_analysis['optimizers'][optimizer]
    axes[0].plot(measurement_steps, entry['log_product'], linewidth=2.5, color=color, label=f'{optimizer.upper()}: log κ(product)')
    axes[0].plot(measurement_steps, entry['sum_log_layers'], '--', linewidth=2, color=color, label=f'{optimizer.upper()}: Σ log κ(layer)')
    axes[1].plot(measurement_steps, entry['upper_bound_gap'], linewidth=2.5, color=color, label=f'{optimizer.upper()}: Σ log κ(layer) - log κ(product)')

axes[0].set_title(f'Condition-number trajectories at representative depth L={analysis_depth}')
axes[0].set_xlabel('Optimization step')
axes[0].set_ylabel('Log-scaled quantity')
axes[0].legend(loc='best', fontsize=9)

axes[1].axhline(0.0, color='black', linewidth=1)
axes[1].set_title('Upper-bound gap (should remain non-negative)')
axes[1].set_xlabel('Optimization step')
axes[1].set_ylabel('Gap')
axes[1].legend(loc='best', fontsize=9)

fig.tight_layout()
plt.show()

condition_fit_rows = []
for optimizer in ['sgd', 'muon']:
    entry = condition_analysis['optimizers'][optimizer]
    product_fit = entry['product_fit']
    sum_fit = entry['sum_log_layers_fit']
    condition_fit_rows.append({
        'optimizer': optimizer.upper(),
        'upper_bound_respected': entry['upper_bound_respected'],
        'log_kappa_product_slope': np.nan if product_fit is None else product_fit['slope'],
        'log_kappa_product_r2': np.nan if product_fit is None else product_fit['r_squared'],
        'sum_log_kappa_layers_slope': np.nan if sum_fit is None else sum_fit['slope'],
        'sum_log_kappa_layers_r2': np.nan if sum_fit is None else sum_fit['r_squared'],
    })

condition_fit_df = pd.DataFrame(condition_fit_rows)
display(condition_fit_df.round(6))


**Interpretation.** In this run, the upper-bound relation is respected for both optimizers. The main role of the diagnostic is therefore cautionary and descriptive: it can reveal whether conditioning is growing, and whether Muon and SGD differ in that growth pattern, but it should not be used to justify a claim that product conditioning must outpace the summed layer conditioning. The notebook now states this relation correctly.


## Secondary diagnostic: log(advantage) versus log(step)

This diagnostic is retained because it existed in the script, but it should be interpreted conservatively. It is a fit through the six measurement times, not an asymptotic time-complexity analysis, and in the current deterministic run it is not the cleanest supporting signal.


In [ ]:
time_rows = []
for depth in config['depths']:
    entry = time_scaling['per_depth'][depth]
    time_rows.append({
        'depth': depth,
        'loglog_slope': entry['slope'],
        'slope_over_depth': entry['ratio_to_depth'],
        'r_squared': entry['r_squared'],
    })

time_df = pd.DataFrame(time_rows)
display(time_df.round(4))
print(f"Slope-of-slopes versus depth: {time_scaling['slope_of_slopes']:.4f}")


## Final verdict and limitations

The conclusion below is intentionally tied to the actual metrics computed by the script and the figures shown in this notebook. It explicitly separates what the deterministic run supports from what remains unproven.


In [ ]:
final_advantages = summary_df[['depth', f'advantage_step_{primary["final_measurement_step"]}']].copy()
max_row = final_advantages.iloc[final_advantages[f'advantage_step_{primary["final_measurement_step"]}'].idxmax()]

limitations = [
    'single deterministic target/data draw and deterministic initialization rule',
    'fixed 300-step training horizon',
    'heuristic stable-LR baseline for SGD rather than globally optimized SGD',
    'product-norm curvature proxy rather than a true Hessian spectral computation',
    'no uncertainty estimates across seeds and no asymptotic proof',
]

limitations_md = '\n'.join([f'- {item}' for item in limitations])

conclusion_md = f"""
### Verdict

**{verdict['label']}.** {verdict['summary']}

### What the deterministic run actually shows

- Final-step advantage ratios at step {primary['final_measurement_step']} increase from shallow to deep settings, with the largest value in this run at **depth {int(max_row['depth'])}**.
- The semilog fit of final-step advantage versus depth has slope **{primary['slope']:.4f}**, base-per-layer **{primary['exp_base_per_layer']:.4f}**, and **R² = {primary['r_squared']:.4f}**.
- The threshold-step diagnostics add nuance: shallow-depth loose-threshold behavior can favor SGD, whereas stricter-threshold/deeper-depth behavior is more favorable to Muon.
- The condition-number diagnostic is descriptive and bound-aware, not a proof that Muon has isolated or eliminated all layer coupling.

### What this notebook does not establish

This notebook does **not** prove complexity-class separation, does not identify asymptotic sample or iteration complexity, and does not provide robustness statistics across random seeds.

### Limitations
{limitations_md}
"""

display(Markdown(conclusion_md))
